In [2]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt

import plotly.io as pio
pio.renderers.default = "vscode"
pio.templates.default = "ggplot2"

color_seq = px.colors.qualitative.Dark24
color_seq_train = px.colors.qualitative.Pastel
color_seq_val = px.colors.qualitative.Set1

from glob import glob

prefix = "../../experiment_data/multifield"
csvs = ["margin_join_simpl.csv"]

dfs = [pd.read_csv(f"{prefix}/{csv}") for csv in csvs]

for i, df in enumerate(dfs):
    df.drop(columns=["Unnamed: 0"], inplace=True, errors='ignore')
    df["run"] = csvs[i].split("_")[-1].split(".csv")[0]
    df["name"] = csvs[i].split(".csv")[0]
    
data = pd.concat(dfs, ignore_index=True)
data.head()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


,Simplification Threshold,Count,run,name
0,0.000000,320,simpl,margin_join_simpl
1,0.003033,319,simpl,margin_join_simpl
2,0.007184,318,simpl,margin_join_simpl
3,0.007411,317,simpl,margin_join_simpl
4,0.007487,316,simpl,margin_join_simpl


In [6]:
data = data.sort_values(by=["run", "Count"], ascending=False)
fig = make_subplots(rows=1, cols=1, subplot_titles=["ResNet-MNIST Margin Simplification"], shared_yaxes=True)

accuracies = [98.1, 98.13, 98.17]

for i, (e, df) in enumerate(data.groupby(["run"])):
	df = df.sort_values(by="Count", ascending=False)

	trace = go.Scatter(x = df["Simplification Threshold"], y = df["Count"], mode="lines", line=dict(shape="hv"), name=f"Run {e[0].capitalize()} ({accuracies[i]}%)", legendgrouptitle=dict(text="Run (Accuracy%)"), line_color=color_seq[i])

	fig.add_trace(
		trace,
		row=1,
		col=1,
	)

fig.update_annotations(font_size=16)
fig.update_layout(margin=dict(l=0, r=0, t=30, b=0), width=300*2, height=180*2, font=dict(size=14), showlegend=False, legend=dict(
	xanchor="right", yanchor="top", x=0.98, y=0.98, bgcolor="rgba(255,255,255,0.5)", font=dict(size=14)))
fig.update_xaxes(title_text="Simplification Threshold", title_standoff=18, automargin=True)
fig.update_yaxes(title_text="#Valleys", type="log", title_standoff=18, automargin=True)
fig.show()
fig.write_image(f"plots/margin_simpl.png", scale=4)